# Day 17: SQL 字符串函数与日期处理 —— 数据清洗必备

> **目标**: 掌握 SQL 字符串和日期函数，理解数据清洗中 80% 的脏数据问题都可以用这两个类函数解决。
> **环境**: DuckDB (`%%sql`)
> **数据**: `../data/sales.csv` + `../data/customers.csv`

In [1]:
import duckdb

%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

## 1. 字符串函数速查

| 函数 | 作用 | DuckDB 示例 | Python 等价 |
|------|------|-------------|-------------|
| `UPPER` | 转大写 | `UPPER(country)` | `df['col'].str.upper()` |
| `LOWER` | 转小写 | `LOWER(email)` | `df['col'].str.lower()` |
| `LENGTH` | 长度 | `LENGTH(product)` | `df['col'].str.len()` |
| `TRIM` | 去两端空格 | `TRIM('  abc  ')` | `df['col'].str.strip()` |
| `LTRIM/RTRIM` | 去左/右空格 | `LTRIM(col)` | `df['col'].str.lstrip()` |
| `SUBSTR` | 子串 | `SUBSTR(order_id, 2, 3)` | `df['col'].str[1:4]` |
| `REPLACE` | 替换 | `REPLACE(country, 'US', 'USA')` | `df['col'].str.replace(...)` |
| `CONCAT` | 拼接 | `CONCAT(first, '-', last)` | `df['a'] + '-' + df['b']` |
| `||` | 拼接简写 | `first || '-' || last` | |
| `LIKE` | 模糊匹配 | `product LIKE '%Phone%'` | `.str.contains('Phone')` |
| `ILIKE` | 不区分大小写模糊匹配 | `product ILIKE '%phone%'` | `.str.contains('phone', case=False)` |
| `STRFTIME` | 格式化日期 | `STRFTIME('%Y-%m', order_date)` | `.dt.strftime('%Y-%m')` |

⚠️ **注意**: `SUBSTR(string, start, length)` —— start 从 1 开始，不是 0。

## 2. 字符串函数实战

### 2.1 大小写转换与去空格

数据清洗第一步：统一大小写、去除多余空格。

In [2]:
%%sql
-- 把国家名统一大写
SELECT DISTINCT UPPER(country) AS country_upper
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

country_upper
FRANCE
CHINA
GERMANY
UK
US


In [3]:
%%sql
-- 把产品名统一小写并去两端空格
SELECT DISTINCT TRIM(LOWER(product)) AS product_clean
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

product_clean
mouse
headphones
laptop
phone
monitor


### 2.2 子串提取与替换

提取 `order_id` 中的数字部分，替换国家名。

In [4]:
%%sql
-- 提取 order_id 的数字部分（O1000 → 1000）
SELECT order_id,
       SUBSTR(order_id, 2, 10) AS order_num,
       CAST(SUBSTR(order_id, 2, 10) AS INTEGER) AS order_num_int
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,order_num,order_num_int
O1000,1000,1000
O1001,1001,1001
O1002,1002,1002
O1003,1003,1003
O1004,1004,1004


In [5]:
%%sql
-- 把 US 替换为 USA
SELECT country, REPLACE(country, 'US', 'USA') AS country_new
FROM '../data/sales.csv'
WHERE country = 'US'
LIMIT 5;

Running query in 'duckdb:///:memory:'

country,country_new
US,USA
US,USA
US,USA
US,USA
US,USA


### 2.3 拼接与模糊匹配

拼接多列，模糊筛选产品。

In [6]:
%%sql
-- 拼接客户ID和国家
SELECT customer_id, country,
       CONCAT(customer_id, ' - ', country) AS label,
       customer_id || '_' || country AS label2
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

customer_id,country,label,label2
C007,Germany,C007 - Germany,C007_Germany
C004,US,C004 - US,C004_US
C005,US,C005 - US,C005_US
C007,US,C007 - US,C007_US
C003,France,C003 - France,C003_France


In [7]:
%%sql
-- 模糊匹配：含 Phone 或 Laptop 的产品（不区分大小写）
SELECT order_id, product
FROM '../data/sales.csv'
WHERE product ILIKE '%Phone%' OR product ILIKE '%Laptop%'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,product
O1002,Laptop
O1003,Headphones
O1004,Phone
O1005,Laptop
O1006,Phone


## 3. 日期函数速查

| 函数 | 作用 | DuckDB 示例 | Python 等价 |
|------|------|-------------|-------------|
| `STRFTIME` | 日期格式化 | `STRFTIME('%Y-%m', date)` | `.dt.strftime('%Y-%m')` |
| `EXTRACT` | 提取日期部分 | `EXTRACT(YEAR FROM date)` | `.dt.year` |
| `EXTRACT` | 提取月份 | `EXTRACT(MONTH FROM date)` | `.dt.month` |
| `EXTRACT` | 提取星期几 | `EXTRACT(DOW FROM date)` | `.dt.dayofweek` |
| `DATE_TRUNC` | 截断到粒度 | `DATE_TRUNC('month', date)` | |
| `CURRENT_DATE` | 当前日期 | `CURRENT_DATE` | `pd.Timestamp.now()` |
| `DATE_DIFF` | 日期间隔 | `DATE_DIFF('day', d1, d2)` | `(d2 - d1).days` |

⚠️ **DuckDB 读 CSV 日期**: CSV 读入的日期默认是字符串，需要先 `CAST` 或用 `STRPTIME` 转换。

## 4. 日期函数实战

### 4.1 日期格式化与提取

CSV 的日期是字符串，需要显式处理。

In [8]:
%%sql
-- 日期格式化：提取年月
SELECT order_id,
       order_date,
       STRFTIME('%Y-%m', order_date) AS year_month,
       EXTRACT(YEAR FROM order_date) AS year,
       EXTRACT(MONTH FROM order_date) AS month
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,order_date,year_month,year,month
O1000,2024-01-01,2024-01,2024,1
O1001,2024-01-01,2024-01,2024,1
O1002,2024-01-02,2024-01,2024,1
O1003,2024-01-03,2024-01,2024,1
O1004,2024-01-03,2024-01,2024,1


In [9]:
%%sql
-- 按年月分组统计
SELECT STRFTIME('%Y-%m', order_date) AS ym,
       COUNT(*) AS cnt,
       SUM(total) AS total_sales
FROM '../data/sales.csv'
GROUP BY ym
ORDER BY ym;

Running query in 'duckdb:///:memory:'

ym,cnt,total_sales
2024-01,43,99365
2024-02,40,81795
2024-03,42,133371
2024-04,41,123882
2024-05,42,71176
2024-06,41,119061
2024-07,43,101271
2024-08,42,104388
2024-09,41,103183
2024-10,42,117265


### 4.2 日期差与截断

计算距今天数、按月份截断。

In [10]:
%%sql
-- 计算每个订单距今（2024-06-01）的天数
SELECT order_id,
       order_date,
       DATE_DIFF('day', order_date, CAST('2024-06-01' AS DATE)) AS days_ago
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,order_date,days_ago
O1000,2024-01-01,152
O1001,2024-01-01,152
O1002,2024-01-02,151
O1003,2024-01-03,150
O1004,2024-01-03,150


In [11]:
%%sql
-- 按月份截断（归到月初）
SELECT order_id,
       order_date,
       DATE_TRUNC('month', order_date) AS month_start
FROM '../data/sales.csv'
LIMIT 5;

Running query in 'duckdb:///:memory:'

order_id,order_date,month_start
O1000,2024-01-01,2024-01-01 00:00:00
O1001,2024-01-01,2024-01-01 00:00:00
O1002,2024-01-02,2024-01-01 00:00:00
O1003,2024-01-03,2024-01-01 00:00:00
O1004,2024-01-03,2024-01-01 00:00:00


## 5. 综合场景 —— 清洗 + 分析

**场景**: 清洗 `product` 列（去空格+转小写），按月份统计每个清洗后产品的销售额。

In [12]:
%%sql
SELECT
    TRIM(LOWER(product)) AS product_clean,
    STRFTIME('%Y-%m', order_date) AS ym,
    COUNT(*) AS order_cnt,
    SUM(total) AS total_sales
FROM '../data/sales.csv'
GROUP BY product_clean, ym
ORDER BY ym, total_sales DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

product_clean,ym,order_cnt,total_sales
phone,2024-01,13,30155
mouse,2024-01,5,21482
laptop,2024-01,8,16074
monitor,2024-01,6,15684
keyboard,2024-01,5,8889
headphones,2024-01,6,7081
mouse,2024-02,9,23972
keyboard,2024-02,6,21577
phone,2024-02,7,18382
monitor,2024-02,9,8981


## 今日要点总结

| 场景 | SQL | Python |
|------|-----|--------|
| 转大写 | `UPPER(col)` | `df['col'].str.upper()` |
| 转小写 | `LOWER(col)` | `df['col'].str.lower()` |
| 去空格 | `TRIM(col)` | `df['col'].str.strip()` |
| 长度 | `LENGTH(col)` | `df['col'].str.len()` |
| 子串 | `SUBSTR(col, 2, 3)` | `df['col'].str[1:4]` |
| 替换 | `REPLACE(col, 'a', 'b')` | `df['col'].str.replace('a', 'b')` |
| 拼接 | `CONCAT(a, '-', b)` | `df['a'] + '-' + df['b']` |
| 模糊匹配 | `col LIKE '%x%'` | `df['col'].str.contains('x')` |
| 年月提取 | `STRFTIME('%Y-%m', date)` | `df['date'].dt.strftime('%Y-%m')` |
| 年提取 | `EXTRACT(YEAR FROM date)` | `df['date'].dt.year` |
| 月提取 | `EXTRACT(MONTH FROM date)` | `df['date'].dt.month` |
| 日期差 | `DATE_DIFF('day', d1, d2)` | `(d2 - d1).days` |

**核心心法**: SQL 字符串/日期函数 = Pandas `.str`/`.dt` 的 SQL 版本。数据清洗中，先统一格式（大小写/空格/日期格式），再分析。